# Importer pakker

In [1]:
import sys

WORK_TREE = "/.claude/worktrees/sad-nash"
# WORK_TREE = ""
sys.path.insert(0, "C:/Users/DstMove/Desktop/claude/projects/sutlab" + WORK_TREE)

from pathlib import Path
import pandas as pd
import sutlab

In [2]:
pd.set_option("display.max_rows", 100)

# Indlæs data

## Konfigurer indlæsning

**Konfigurationen skal tilpasses til den data, du vil indlæse.**

Stier til TA filer, måltotaler og metadata:

In [3]:
# Sti til mappe med TA filer på langt format
SUT_DATA_DIR = Path("c:/users/dstmove/desktop/claude/projects/sutlab/data/examples")

# Navn på årlige TA filer
SUT_FILE = "ta_l_{YYYY}.parquet"

# Sti til mappe med måltotaler på langt format
TARGETS_DATA_DIR = Path("c:/users/dstmove/desktop/claude/projects/sutlab/data/examples")

# Navn på årlige måltotal filer
TARGETS_FILE = "ta_targets_l_{YYYY}.xlsx"

# Sti til excelfiler med TA metadata
METADATA_DIR = Path("c:/users/dstmove/desktop/claude/projects/sutlab/data/examples/metadata")

# Navn på kolonne navns metadata fil
METADATA_COLUMNS_FILE = "ta_columns.xlsx"

# Navn på klassifikations metadata fil
METADATA_CLASSIFICATIONS_FILE = "ta_classifications.xlsx"

År som data skal indlæses for:

In [4]:
START_YEAR = 2008
END_YEAR = 2019

## Beregning af stier

Beregn stier:

In [5]:
# Beregning af liste med alle år
YEARS = list(range(START_YEAR, END_YEAR + 1))

# Stier til metadata
METADATA_COLUMNS_PATH = METADATA_DIR / METADATA_COLUMNS_FILE
METADATA_CLASSIFICATIONS_PATH = METADATA_DIR / METADATA_CLASSIFICATIONS_FILE

# Stier til TA'er
SUT_FILE_LIST = [SUT_FILE.format(YYYY=year) for year in YEARS]
SUT_DATA_PATH_LIST = [SUT_DATA_DIR / sut_file for sut_file in SUT_FILE_LIST]

# Stier til måltotaler
TARGETS_FILE_LIST = [TARGETS_FILE.format(YYYY=year) for year in YEARS]
TARGETS_DATA_PATH_LIST = [TARGETS_DATA_DIR / targets_file for targets_file in TARGETS_FILE_LIST]

Vis stier:

In [6]:
print(f"Kolonne navns metadata indlæses fra: {METADATA_COLUMNS_PATH}")
print(f"Klassifikations metadata indlæses fra: {METADATA_CLASSIFICATIONS_PATH}")
print("\n")

print(f"Følgende TA filer indlæses fra mappen: {SUT_DATA_DIR}")
for sut_file in SUT_FILE_LIST :
    print(f"  {sut_file}")
print("\n")

print(f"Følgende måltotal filer indlæses fra mappen: {TARGETS_DATA_DIR}")
for targets_file in TARGETS_FILE_LIST :
    print(f"  {targets_file}")

Kolonne navns metadata indlæses fra: c:\users\dstmove\desktop\claude\projects\sutlab\data\examples\metadata\ta_columns.xlsx
Klassifikations metadata indlæses fra: c:\users\dstmove\desktop\claude\projects\sutlab\data\examples\metadata\ta_classifications.xlsx


Følgende TA filer indlæses fra mappen: c:\users\dstmove\desktop\claude\projects\sutlab\data\examples
  ta_l_2008.parquet
  ta_l_2009.parquet
  ta_l_2010.parquet
  ta_l_2011.parquet
  ta_l_2012.parquet
  ta_l_2013.parquet
  ta_l_2014.parquet
  ta_l_2015.parquet
  ta_l_2016.parquet
  ta_l_2017.parquet
  ta_l_2018.parquet
  ta_l_2019.parquet


Følgende måltotal filer indlæses fra mappen: c:\users\dstmove\desktop\claude\projects\sutlab\data\examples
  ta_targets_l_2008.xlsx
  ta_targets_l_2009.xlsx
  ta_targets_l_2010.xlsx
  ta_targets_l_2011.xlsx
  ta_targets_l_2012.xlsx
  ta_targets_l_2013.xlsx
  ta_targets_l_2014.xlsx
  ta_targets_l_2015.xlsx
  ta_targets_l_2016.xlsx
  ta_targets_l_2017.xlsx
  ta_targets_l_2018.xlsx
  ta_targets_l_

## Indlæs data

Indlæs metadata:

In [7]:
metadata = sutlab.load_metadata_from_excel(
    columns_path = METADATA_COLUMNS_PATH,
    classifications_path = METADATA_DIR / METADATA_CLASSIFICATIONS_FILE,
)

In [8]:
nrnr = metadata.classifications.products.copy()

In [23]:
sort_illegal = nrnr[nrnr["nrnr"].str[0].eq("H")].copy().assign(note="Sort og illegal produktion")
egen_soft_forsk = nrnr[nrnr["nrnr"].str[:5].isin(["K6201","K7201","K7209"])].copy().assign(note="Egenproduceret software og forskning")
rep_ved = nrnr[nrnr["nrnr"].isin(["M000002","M330000","M420000","M430000","M452000"])].copy().assign(note="Reperation og vedligehold")
offent_ikke_mark_konsum = nrnr[nrnr["nrnr"].str[0].eq("Q")].copy().assign(note="Offentlige ikke-markedsmæssige tjenester til forbrug")
s_forsk = nrnr[nrnr["nrnr"].isin(["S720142","S720992"])].copy().assign(note="Offentligt salg af forskning")
licens_soft = nrnr[nrnr["nrnr"].isin(["T000005"])].copy().assign(note="Licensbetaling")
fin_tjen_nrnr = [
    "T641020",
    "T641100",
    "T641101",
    "T641900",
    "T641901",
    "T642010",
    "T643010",
    "T643039",
    "T649100",
    "T649101",
    "T649201",
    "T649203",
    "T649204",
    "T649900"
]
fin_tjen = nrnr[nrnr["nrnr"].isin(fin_tjen_nrnr)].copy().assign(note="Finansielle tjenester")
forsikring = nrnr[nrnr["nrnr"].isin(["T651201","T651203"])].copy().assign(note="Skadesforsikring")
royalties = nrnr[nrnr["nrnr"].str[:5].isin(["T9000"])].copy().assign(note="Royalties")
fv_lager = nrnr[nrnr["nrnr"].isin(["U002065"])].copy().assign(note="Færdigvarer lager")
orig_vaerker = nrnr[nrnr["nrnr"].str[:5].isin(["U9000"])].copy().assign(note="Originale værker")
forsyning = nrnr[nrnr["nrnr"].str[:3].isin(["U35","U36"])].copy().assign(note="Forsyning")
energi = nrnr[nrnr["nrnr"].str[:3].isin(["V27"])].copy().assign(note="Energi")

laaste = pd.concat([sort_illegal,egen_soft_forsk,rep_ved,offent_ikke_mark_konsum,s_forsk,licens_soft,fin_tjen,forsikring,royalties,fv_lager,orig_vaerker,forsyning,energi]).reset_index(drop=True)
laaste = laaste.sort_values("nrnr").reset_index(drop=True)

In [ ]:
# laaste.to_excel("C:/Users/DstMove/Desktop/claude/projects/sutlab/notebooks/laaste_nrnr.xlsx", index=False)